# Rainfall Analysis and Planting Advisory

This notebook cleans the Machakos rainfall readings, analyses monthly rainfall patterns for five Kenyan counties using NumPy, and generates planting-window recommendations.

In [3]:
import csv
import numpy as np

counties = np.array([
    "Nairobi",
    "Nakuru",
    "Kisumu",
    "Mombasa",
    "Machakos"
])

months = np.array([
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

In [4]:
def clean_machakos(raw_values, month_names):
    valid_values = []
    invalid_indices = []

    for index, value in enumerate(raw_values):
        try:
            valid_values.append(int(value))
        except ValueError:
            invalid_indices.append(index)

    imputation_value = round(sum(valid_values) / len(valid_values))

    cleaned_values = []
    for index, value in enumerate(raw_values):
        try:
            cleaned_values.append(int(value))
        except ValueError:
            cleaned_values.append(imputation_value)

    imputed_months = month_names[invalid_indices]

    print("Machakos valid-value average:", imputation_value)
    print("Imputed months:", ", ".join(imputed_months))

    return cleaned_values


machakos_clean = clean_machakos(machakos_raw, months)
print("Cleaned Machakos row:", machakos_clean)

Machakos valid-value average: 68
Imputed months: April, September
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Combined Rainfall Array

In [5]:
R = np.array(clean_rows + [machakos_clean])

print(R)
print("Shape:", R.shape)
print("Data type:", R.dtype)

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Data type: int64


## Rainfall Statistics

In [6]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("Annual rainfall totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nAverage rainfall by month:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

Annual rainfall totals:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average rainfall by month:
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


In [7]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = months[wettest_indices]

print("Wettest month by county:")
for county, wettest_month in zip(counties, wettest_months):
    print(f"{county}: {wettest_month}")

Wettest month by county:
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


In [8]:
dry_mask = R < 50
dry_month_counts = dry_mask.sum(axis=1)

print("Number of dry months by county:")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")

Number of dry months by county:
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


In [9]:
consecutive_rainfall = R[:, :-1] + R[:, 1:]
planting_start_indices = np.argmax(consecutive_rainfall, axis=1)

planting_windows = np.array([
    f"{months[index]}-{months[index + 1]}"
    for index in planting_start_indices
])

highest_two_month_totals = np.max(consecutive_rainfall, axis=1)

print("Recommended planting windows:")
for county, window, total in zip(
    counties,
    planting_windows,
    highest_two_month_totals
):
    print(f"{county}: {window} ({total} mm combined)")

Recommended planting windows:
Nairobi: April-May (395 mm combined)
Nakuru: April-May (265 mm combined)
Kisumu: April-May (420 mm combined)
Mombasa: April-May (390 mm combined)
Machakos: November-December (278 mm combined)


In [10]:
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)

    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    for row in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow([
            row[0],
            int(row[1]),
            row[2],
            int(row[3]),
            row[4]
        ])

print("rainfall_summary.csv created successfully.")

rainfall_summary.csv created successfully.


In [11]:
with open("rainfall_summary.csv", "r", encoding="utf-8") as csv_file:
    print(csv_file.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December



## Planting Advisory

The NGO should advise farmers in Nairobi, Nakuru, Kisumu, and Mombasa to prepare for planting during the April-May window, which has the highest combined rainfall in those counties. Machakos has a different pattern, with November-December providing its strongest two-month rainfall window. Kisumu records the highest annual rainfall and no months below 50 mm, while Machakos has the lowest annual total and six dry months. Farmers should combine these recommendations with current weather forecasts and local soil conditions before planting. The Machakos results should be treated with caution because the April and September readings were invalid and were replaced with the rounded average of the valid Machakos observations.

## Data Quality Notes

The Machakos source row contained `"N/A"` for April and an empty string for September. Both invalid readings were replaced with 68 mm, the rounded average of the ten valid Machakos readings. Mean imputation prevents the observations from being discarded, but it may reduce the apparent variation in Machakos rainfall. The dataset also contains only one annual cycle, so the planting recommendations should not be treated as long-term climate normals.